In [ ]:
import csv

# Part 1
total_logins = 0
success_logins = 0
failed_logins = 0

failed_counts = {}
failed_ips = {}
failed_systems = {}
user_ips = {}

suspicious_rows_buffer = []

**Part 1: Initialization & In-Memory Storage**
  * Integer counters are initialized to `0` as a neutral starting point. Each time a matching record is found, the counter increments by 1 — giving an accurate running total by the end of the file.
  * Dictionaries are used because we need to associate a **label**(user, IP, or system name) with a **count**. This lets us look up any user's failure count instantly by key instead of searching through a list.
  * `user_ips` needs to map each **user** to their associate **IPs** — a key-value pair that only a dictionary can express. `suspicious_rows_buffer` just needs to hold rows in order for later writing — no lookup needed, so a list is the right tool.
  * Each user's IPs are stored in a set instead of a list to automatically eliminate duplicates. If the same IP hits the same user 50 times, the set records it only once — giving us the count of **unique** attacking IPs per user, which is more meaningful for threat analysis.

In [ ]:
# Part 2
with open("meridian_logs.csv", "r", newline="") as infile: 
    logins = csv.DictReader(infile)
    csv_headers = logins.fieldnames

    for row in logins:
        total_logins += 1
        
        if row["status"] == "FAILED":
               failed_logins += 1
               user = row["user"]
               ip = row["ip"]
               system = row["system"]
               
               failed_counts[user] = failed_counts.get(user, 0) + 1
               failed_ips[ip] = failed_ips.get(ip, 0) + 1
               failed_systems[system] = failed_systems.get(system, 0) + 1
               
               if user not in user_ips:
                   user_ips[user] = set()
               user_ips[user].add(ip)
                    
               suspicious_rows_buffer.append(row)

        elif row["status"] == "SUCCESS":
               success_logins += 1
        else:
            print("Unknown Values Detected")
    
    print(f"\nTOTAL LOGINS: {total_logins}")
    print(f"{success_logins} successful vs {failed_logins} failed logins.\n")

**Part 2: Single Pass Streaming Analytics Engine**
* `with open()` automatically closes the file after the block finishes — even if an error occurs mid-read. Without it, an unclosed file can cause data corruption or lock the file from other processes.
* `csv.DictReader` treats the first row as column headers and returns each subsequent row as a dictionary. `csv.reader` treats every row as plain list with no header awareness.
* `login.fieldnames` is saved early because once the file is closed, the reader object loses access to it. Storing it in `csv_headers` lets us use the exact same column order when writing the output CSV in Part 3.
* `failed_counts.get(user, 0)` — looks up the user in the dictionary. If found, it returns their current count. If not found yet, it will return `0` as default. `+1` adds 1 to whatever was returned then `failed_counts[user] =` stores the new value back under that user's key. Same logic applies to `failed_ips[ip]` and `failed_systems[system]`.
* `if user not in user_ips` checks if the user already has a set created. If not, it creates a fresh empty set because `user_ips[user].add(ip)` would throw a `KeyError` trying to add to a set that doesn't exist yet.
* `suspicious_rows_buffer` collects the **FAILED** rows for export, **SUCCESS** rows are counted but not stored for further analysis and `else` print protects the loop for unknown values.
* The summary print sits outside the loop but still inside the `with` block to ensure it only runs once after all the rows are processed, printing the final tallies rather than per-row outputs.

In [ ]:
# Part 3
with open("suspicious_findings.csv", "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    
    for row in suspicious_rows_buffer:
        writer.writerow(row)

**Part 3: Exporting Suspicious Findings to CSV**
* Part 3 opens a second block exclusively for writing. Separating read and write operations keeps the code clean and prevents accidental overwrites while the input file is still being processed. Each `with` block has a single responsibility.
* `w` or `write` creates the file if it doesn't exist or wipes it completely if it does — giving a fresh export every run. On the otherhand, using `a` or `append` just adds new rows below existing data without erasing anything — useful for loggin scenarios where you want to accumulate records overtime.
* We call the `fieldnames=csv_headers` that we saved from `logins.fieldnames` in Part 2 before the file closed. Passing it to `csv.DictWriter` tells the write the exact column names and order to use — ensuring the output CSV matches the structure of the original log file.

In [ ]:
# Part 4
print("SOC TRIAGE REPORT: TOP OFFENDERS & TARGETS")
print("-" * 50)

top_users = sorted(failed_counts, key=failed_counts.get, reverse=True)[:3]
top_ips = sorted(failed_ips, key=failed_ips.get, reverse=True)[:3]
top_systems = sorted(failed_systems, key=failed_systems.get, reverse=True)[:3]

print("TOP TARGETED ACCOUNTS:")
for rank, user in enumerate(top_users, 1):
    print(f"  {rank}. User: '{user}' | Total Failures: {failed_counts[user]}")
    
print("\nTOP ATTACKING IP ADDRESSES:")
for rank, ip in enumerate(top_ips, 1):
    print(f"  {rank}. IP: '{ip}' | Total Failures: {failed_ips[ip]}")

print("\nTOP TARGETED SYSTEMS / ASSETS:")
for rank, sys in enumerate(top_systems, 1):
    print(f"  {rank}. Host: '{sys}' | Total Failures: {failed_systems[sys]}")

print("-" * 50)

target_account = "admin"
if target_account in user_ips:
    print(f"Forensic Audit: '{target_account}' was targeted by {len(user_ips[target_account])} unique IP(s): {user_ips[target_account]}")
print("-" * 50)

**Part 4: Sorting and Reporting Top Offenders**
* `sorted(failed_counts, key=failed_counts.get, reverse=True)[:3]`
sorts dictionary keys by their values in descending order and 
slices the top 3 results — identifying the highest-risk users, 
IPs, and systems without overwhelming the output.
* `enumerate()` with `start=1` provides human-readable ranking 
starting from 1 instead of 0 — making the report immediately 
readable without translation.
* The forensic check on `admin` specifically examines how many 
unique IPs targeted that account. A single IP suggests an 
isolated brute force attempt. Multiple unique IPs suggest a 
distributed attack — potentially a botnet — requiring a 
different and more urgent response from the security team.

## UPDATED SCRIPT WITH RICH FUNCTIONS ##

In [ ]:
import csv
import os

from rich.console import Console
from rich.table import Table
from rich.panel import Panel

def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

console = Console(record=True)

total_logins = 0
success_logins = 0
failed_logins = 0

failed_counts = {}
failed_ips = {}
failed_systems = {}
user_ips = {}

suspicious_rows_buffer = []

with open("portfolio/05_meridian_login_analysis/meridian_logs.csv", "r", newline="") as infile: 
    logins = csv.DictReader(infile)
    csv_headers = logins.fieldnames

    clear_screen()

    console.print(Panel.fit(
        "        MERIDIAN LOGIN ANALYSIS — SOC TRIAGE        ",
        style="bold white on magenta",
        border_style="magenta"
    ))
    console.print()

    for row in logins:
        total_logins += 1
        
        if row["status"] == "FAILED":
               failed_logins += 1
               user = row["user"]
               ip = row["ip"]
               system = row["system"]
               
               failed_counts[user] = failed_counts.get(user, 0) + 1
               failed_ips[ip] = failed_ips.get(ip, 0) + 1
               failed_systems[system] = failed_systems.get(system, 0) + 1
               
               if user not in user_ips:
                   user_ips[user] = set()
               user_ips[user].add(ip)
                    
               suspicious_rows_buffer.append(row)

        elif row["status"] == "SUCCESS":
               success_logins += 1
        else:
            console.print("[yellow]WARNING: Unknown Values Detected[/yellow]")
    
    console.print(f"[bold]TOTAL LOGINS:[/bold] {total_logins}")
    console.print(f"[green] {success_logins} Successful[/green] vs [red] {failed_logins} Failed[/red] logins.\n")

with open("portfolio/05_meridian_login_analysis/suspicious_findings.csv", "w", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    
    for row in suspicious_rows_buffer:
        writer.writerow(row)

console.print("[bold cyan]SOC TRIAGE REPORT: TOP OFFENDERS & TARGETS[/bold cyan]")
console.print("-" * 50)

top_users = sorted(failed_counts, key=failed_counts.get, reverse=True)[:3]
top_ips = sorted(failed_ips, key=failed_ips.get, reverse=True)[:3]
top_systems = sorted(failed_systems, key=failed_systems.get, reverse=True)[:3]

user_table = Table(title="Top Targeted Accounts", title_style="bold yellow", box=None)
user_table.add_column("Rank", justify="center", style="dim")
user_table.add_column("Username", style="bold gold1")
user_table.add_column("Total Failures", justify="right", style="bold red")

for rank, user in enumerate(top_users, 1):
    user_table.add_row(str(rank), f"'{user}'", str(failed_counts[user]))
    
ip_table = Table(title="Top Attacking IP Addresses", title_style="bold red", box=None)
ip_table.add_column("Rank", justify="center", style="dim")
ip_table.add_column("IP Address", style="bold bright_red")
ip_table.add_column("Total Failures", justify="right", style="bold red")

for rank, ip in enumerate(top_ips, 1):
    ip_table.add_row(str(rank), f"'{ip}'", str(failed_ips[ip]))
    
sys_table = Table(title="Top Targeted Systems / Assets", title_style="bold cyan", box=None)
sys_table.add_column("Rank", justify="center", style="dim")
sys_table.add_column("Host", style="bold cyan")
sys_table.add_column("Total Failures", justify="right", style="bold red")

for rank, sys in enumerate(top_systems, 1):
    sys_table.add_row(str(rank), f"'{sys}'", str(failed_systems[sys]))

console.print(user_table)
console.print()
console.print(ip_table)
console.print()
console.print(sys_table)
console.print("="*50)

target_account = "admin"
ALERT_THRESHOLD = 5

if target_account in user_ips:
    total_admin_failures = failed_counts[target_account]
    unique_ip_count = len(user_ips[target_account])
    formatted_ips = ", ".join([f"[bold white]{ip}[/bold white]" for ip in user_ips[target_account]])

    if total_admin_failures >= ALERT_THRESHOLD:
        
        if unique_ip_count >= 3:
            attack_profile = "[bold bright_red]CRITICAL DISTRIBUTED ATTACK[/bold bright_red]"
            border_color = "bright_red"
        else:
            attack_profile = "[bold orange3]CRITICAL ISOLATED ATTACK[/bold orange3]"
            border_color = "orange3"

        console.print(Panel(
                f"Threat Profile: {attack_profile}\n\n"
                f"Account [yellow]'{target_account}'[/yellow] was hammered [bold red]{total_admin_failures}[/bold red] times!\n"
                f"Source Vector: Originated from [bold white]{unique_ip_count}[/bold white] unique IP address(es).\n"
                f"Rogue IP List: {formatted_ips}",
                title="Forensic Audit Alert",
                border_style=border_color
            ))
    else:
        console.print(f"[green]Audit: Low-volume threat detected on '{target_account}' ({total_admin_failures} failure). No action needed at this time.[/green]")
else:
    console.print(f"[green]Forensic Audit: No malicious footprint detected on target account '{target_account}'.[/green]")

console.save_text("portfolio/05_meridian_login_analysis/soc_triage_report.txt")
console.save_html("portfolio/05_meridian_login_analysis/soc_triage_report.html")

**Second Iteration:**
* Added `rich` library functions (`Console, Table, and Panel`) along with `MONOKAI` from `rich.terminal_theme`.
* Updated the forensic check logic on `admin`:
    * Added a constant variable `ALERT_THRESHOLD` to hold the alert from setting off until it reaches the limit of **5** failures, filtering out low-volume noise like legitimate password typos.
    * Implemented dynamic threat profiling by measuring the size of the unique IP set (`len(user_ips[target_account])`):
        * If targeted by **3 or more unique IPs**, it flags a `CRITICAL DISTRIBUTED ATTACK` in a bright red panel (indicating a potential botnet or password-spray vector).
        * If targeted by **1-2 unique IPs**, it flags a `CRITICAL ISOLATED ATTACK` in an orange panel (indicating a targeted brute-force attempt).
* Implemented a Dual Reporting Pipeline:
    * Initialized the `Console` with `record=True` to capture terminal stdout into an in-memory buffer.
    * Executed `console.save_text()` first to capture a clean plain-text backup.
    * Followed up with `console.save_html(theme=MONOKAI)` to export a fully formatted dark-mode dashboard without clearing the recording buffer prematurely.